# 01. Adquisicion y preparacion de datos

## Situacion problematica

ACRE Africa ofrece seguros agricolas a pequenos productores de Africa oriental. Cuando un
agricultor presenta una reclamacion, un evaluador revisa las fotografias que el propio
agricultor toma con su telefono, decide si el cultivo esta danado, identifica la causa y estima
el porcentaje de perdida; de esa estimacion depende el monto que se paga. Esa revision manual es
el cuello de botella del proceso y la sequia concentra la mayoria de las reclamaciones.

El proyecto Eyes on the Ground construyo un conjunto de datos etiquetado para entrenar modelos
que sustituyan esa revision. Los modelos ajustados sobre las primeras temporadas alcanzaron un
desempeno aceptable que no se sostuvo al aplicarlos a una temporada posterior.

## Problema cientifico

Identificar que caracteristicas del conjunto de datos limitan la generalizacion entre temporadas
y que preprocesamiento permite estimar de forma confiable la magnitud del dano por sequia a
partir de fotografias.

## Objetivos especificos

1. Describir la estructura y la calidad de las variables tabulares y de las imagenes,
   cuantificando valores faltantes, duplicados e inconsistencias entre el tipo de dano declarado
   y su magnitud.
2. Comparar la distribucion de las variables clave entre temporadas y entre los conjuntos de
   entrenamiento y prueba, midiendo la magnitud del desplazamiento de dominio.
3. Cuantificar la agrupacion de imagenes por campo y su efecto sobre el riesgo de fuga de
   informacion al particionar los datos.

Este primer cuaderno documenta la obtencion reproducible del conjunto, describe sus variables y
deja registradas las decisiones de limpieza antes del analisis exploratorio.

## Preparacion del entorno

La primera celda localiza la raiz del proyecto subiendo desde el directorio de trabajo hasta el
primer directorio que contiene una carpeta `src`, y la agrega a `sys.path`. Asi el cuaderno
funciona sin importar desde donde se inicie Jupyter.

In [ ]:
import sys
from pathlib import Path


def raiz_proyecto():
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "src").is_dir():
            return candidato
    raise RuntimeError("No se encontro la raiz del proyecto")


RAIZ = raiz_proyecto()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

from src import carga, descarga, limpieza, tablas
from src.config import CONTEO_OFICIAL_TEMPORADA, ETAPAS_CRECIMIENTO, TIPOS_DANO

## Obtencion reproducible de los datos

Los datos se descargan con `src.descarga`, que trae los CSV desde la API de Zindi y las imagenes
desde Google Drive; las instrucciones completas estan en el `README.md`. La verificacion siguiente
confirma el estado local de los archivos sin volver a descargar nada.

In [ ]:
descarga.imprimir_verificacion(descarga.verificar())

El conteo local de imagenes por temporada queda por debajo del conteo oficial documentado por la
competencia. La siguiente tabla cuantifica la diferencia temporada por temporada.

In [ ]:
train = carga.cargar_train()
test = carga.cargar_test()

conteo_local = train["temporada"].value_counts().reindex(CONTEO_OFICIAL_TEMPORADA.keys())
comparacion_temporada = pd.DataFrame(
    {"oficial": pd.Series(CONTEO_OFICIAL_TEMPORADA), "local": conteo_local}
)
comparacion_temporada["diferencia"] = comparacion_temporada["local"] - comparacion_temporada["oficial"]
comparacion_temporada

La correspondencia entre `Train.csv` y las imagenes en disco es exacta: no hay registros sin
imagen ni imagenes sin registro. Por lo tanto la diferencia frente al conteo oficial no proviene
de una descarga incompleta, sino de que el conjunto distribuido contiene unos pocos registros
menos que la cifra documentada. Se trabaja con el conjunto tal como se distribuye y se deja
constancia de la diferencia.

## Lectura de los conjuntos y variables derivadas

Las columnas originales del CSV se renombran al espanol: `ID` a identificador, `filename` a
archivo, `growth_stage` a etapa, `damage` a dano, `extent` a magnitud y `season` a temporada. El
nombre de archivo no es arbitrario sino un identificador estructurado con dos convenciones: un
formato codificado (`L<productor>F<campo>C<cultivo>S<sitio><tipo><secuencia>.jpg`) usado por tres
temporadas, y un formato secuencial (`<productor>_<tipo>_<n>_<resto>.JPG`) usado por la temporada
mas antigua. De ambos se derivan el productor, el campo, el cultivo, el sitio y el tipo de
captura. El identificador de campo es clave porque las fotografias de un mismo campo son
observaciones dependientes, y repartirlas al azar entre entrenamiento y validacion filtraria
informacion.

In [ ]:
print(f"train: {train.shape[0]} registros, {train.shape[1]} columnas")
print(f"test:  {test.shape[0]} registros, {test.shape[1]} columnas")
train.head()

`Test.csv` no incluye la columna de magnitud porque ese es precisamente el valor que la
competencia pide estimar; si trae el tipo de dano declarado, lo que sera relevante para acotar el
problema mas adelante.

## Diccionario de datos

El diccionario resume, para cada variable, su tipo, su escala de medicion, la cantidad de valores
presentes y ausentes, el numero de valores distintos y un ejemplo.

In [ ]:
carga.diccionario_datos(train)

Los codigos de etapa de crecimiento y de tipo de dano son abreviaturas de dos y tres letras. Los
catalogos de referencia permiten interpretarlas.

In [ ]:
catalogo_etapa = pd.DataFrame(ETAPAS_CRECIMIENTO.items(), columns=["codigo", "descripcion"])
catalogo_dano = pd.DataFrame(TIPOS_DANO.items(), columns=["codigo", "descripcion"])
display(catalogo_etapa)
display(catalogo_dano)

## Limpieza y preprocesamiento

El enfoque es centrado en datos: las operaciones de limpieza se aplican solo cuando hay una razon
documentada. No se imputan valores por el simple hecho de que falten, porque en un conjunto de
etiquetas humanas la ausencia de un valor tambien es informacion sobre el proceso de anotacion.
En lugar de eliminar registros, se agregan indicadores de calidad que permiten filtrar segun el
proposito de cada analisis posterior.

### Valores faltantes

In [ ]:
limpieza.resumen_faltantes(train)

### Duplicados y composicion por tipo de captura

Ademas de revisar duplicados de identificador y de nombre de archivo, se cruzan la temporada con
el formato de nombre y con el tipo de captura. Una composicion desigual del tipo de captura entre
temporadas afecta la comparabilidad, porque cada tipo de captura tiene un proposito distinto y,
como se vera, una magnitud tipica distinta.

In [ ]:
limpieza.duplicados_registro(train)

In [ ]:
tablas.tabla_contingencia(train, "temporada", "formato_nombre")

In [ ]:
tablas.tabla_contingencia(train, "temporada", "tipo_captura")

### Validacion de dominios

Se comprueba que cada variable respete su dominio esperado: la etapa dentro del catalogo de
cuatro valores, el tipo de dano dentro del catalogo de ocho valores, y la magnitud entre cero y
cien y multiplo de diez.

In [ ]:
limpieza.validar_dominios(train)

### Regla estructural entre tipo de dano y magnitud

La magnitud solo deberia tomar valores positivos cuando el dano declarado es sequia. Esta
verificacion documenta el nivel de ruido de etiquetado y condiciona el desempeno maximo
alcanzable por cualquier modelo. Los casos que la rompen no se corrigen: se conservan y se
analizan como ruido.

In [ ]:
limpieza.regla_estructural(train)

### Correspondencia entre registros e imagenes

El analisis posterior combina la tabla de etiquetas con la tabla de atributos de imagen, de modo
que ambos conjuntos deben coincidir. Se cuentan los registros sin imagen en disco y las imagenes
sin registro asociado, para entrenamiento y para prueba.

In [ ]:
for particion, df in (("train", train), ("test", test)):
    sin_imagen = limpieza.registros_sin_imagen(df)
    huerfanas = limpieza.imagenes_huerfanas(df, particion)
    print(f"{particion}: {len(sin_imagen)} registros sin imagen, {len(huerfanas)} imagenes sin registro")

### Bitacora de decisiones

La bitacora consolida todas las verificaciones anteriores junto con la decision tomada en cada
caso, dejando explicito que ninguna elimina informacion.

In [ ]:
limpieza.bitacora(train, "train")

### Marcado de calidad y guardado

Se agregan a cada registro los indicadores `magnitud_valida`, `etiqueta_coherente` y
`apto_modelado`, y se guardan los metadatos de entrenamiento y prueba en formato Parquet. El
cuaderno siguiente parte de estos metadatos y no vuelve a leer los CSV originales.

In [ ]:
train_marcado = limpieza.marcar_calidad(train)
test_marcado = limpieza.marcar_calidad(test)

ruta_train = carga.guardar_metadatos(train_marcado, "train")
ruta_test = carga.guardar_metadatos(test_marcado, "test")

aptos = int(train_marcado["apto_modelado"].sum())
print(f"registros aptos para modelado: {aptos} de {len(train_marcado)}")
print(f"metadatos guardados en:")
print(f"  {ruta_train}")
print(f"  {ruta_test}")

## Cierre de la etapa

El conjunto quedo descrito, verificado contra su fuente original y marcado con indicadores de
calidad, sin eliminar informacion. El cuaderno `02_eda_tabular` retoma los metadatos guardados
para estudiar las etiquetas: la magnitud del dano, las variables categoricas, sus relaciones, y
los dos riesgos que condicionan cualquier modelo posterior, la agrupacion por campo y el
desplazamiento entre temporadas.